# ROI Preprocessing

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

## 2. Import Library

In [ ]:
import re
import csv
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm
from datetime import datetime

## 3. Konfigurasi Path & Parameter

In [ ]:
INPUT_SPLITS = {
    'train/no_tumor' : '/content/drive/MyDrive/Tugas Akhir/FINAL_dataset_preprocessed_test/train/no_tumor',
    'train/tumor'    : '/content/drive/MyDrive/Tugas Akhir/FINAL_dataset_preprocessed_test/train/tumor',
    'val/no_tumor'   : '/content/drive/MyDrive/Tugas Akhir/FINAL_dataset_preprocessed_test/val/no_tumor',
    'val/tumor'      : '/content/drive/MyDrive/Tugas Akhir/FINAL_dataset_preprocessed_test/val/tumor',
    'test/no_tumor'  : '/content/drive/MyDrive/Tugas Akhir/FINAL_dataset_preprocessed_test/test/no_tumor',
    'test/tumor'     : '/content/drive/MyDrive/Tugas Akhir/FINAL_dataset_preprocessed_test/test/tumor',
}

OUTPUT_ROOT      = '/content/drive/MyDrive/Tugas Akhir/FINAL_dataset_roi_preprocessed_test'
MIN_CONTOUR_AREA = 500
PADDING          = 5

## 4. Definisi Fungsi ROI

In [ ]:
def rgb_to_grayscale_paper(img_bgr: np.ndarray) -> np.ndarray:
    # Grayscale Eq.2: 0.299R + 0.587G + 0.114B
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB).astype(np.float32)
    gray = (
        0.299 * img_rgb[:, :, 0]
      + 0.587 * img_rgb[:, :, 1]
      + 0.114 * img_rgb[:, :, 2]
    )
    return gray.astype(np.uint8)


def otsu_roi_crop(
    img_bgr: np.ndarray,
    gray: np.ndarray
) -> tuple[np.ndarray, tuple | None, bool]:
    # Otsu -> bersihkan morfologi -> bounding rect -> crop. bbox=None jika fallback full image.
    H, W = gray.shape

    _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    kernel       = cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5))
    binary_clean = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel)
    binary_clean = cv2.morphologyEx(binary_clean, cv2.MORPH_OPEN, kernel)

    contours, _ = cv2.findContours(binary_clean, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    valid_contours = [c for c in contours if cv2.contourArea(c) > MIN_CONTOUR_AREA]

    if not valid_contours:
        return img_bgr.copy(), None, False

    all_points = np.vstack(valid_contours)
    x, y, w, h = cv2.boundingRect(all_points)

    x1 = max(0, x - PADDING)
    y1 = max(0, y - PADDING)
    x2 = min(W, x + w + PADDING)
    y2 = min(H, y + h + PADDING)

    if (x2 - x1) < 32 or (y2 - y1) < 32:
        return img_bgr.copy(), None, False

    return img_bgr[y1:y2, x1:x2], (x1, y1, x2, y2), True


def extract_image_pair_key(image_stem: str) -> str | None:
    # Key pairing dari nama gambar: image_tumor_mask__[pid]_CT_s[sid]
    m = re.search(r'^image_tumor_mask__(.+)_CT_s(\d+)$', image_stem)
    if not m:
        return None
    return f"{m.group(1)}_CT_{m.group(2)}"


def extract_mask_pair_key(mask_stem: str) -> str | None:
    # Key pairing dari nama mask: mask_tumor__[pid]_CT_m[sid]
    m = re.search(r'^mask_tumor__(.+)_CT_m(\d+)$', mask_stem)
    if not m:
        return None
    return f"{m.group(1)}_CT_{m.group(2)}"


def crop_mask_with_bbox(mask_path: Path, bbox: tuple) -> np.ndarray:
    # Crop mask dengan bbox sama persis dengan gambar pasangannya
    mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    if mask is None:
        raise FileNotFoundError(f"Mask tidak dapat dibaca: {mask_path}")
    x1, y1, x2, y2 = bbox
    H, W = mask.shape
    x1 = max(0, min(x1, W - 1))
    y1 = max(0, min(y1, H - 1))
    x2 = max(1, min(x2, W))
    y2 = max(1, min(y2, H))
    return mask[y1:y2, x1:x2]


def process_image(src_path: Path) -> tuple[np.ndarray, tuple | None, bool, str]:
    img_bgr = cv2.imread(str(src_path))
    if img_bgr is None:
        return None, None, False, "ERROR_READ"
    gray = rgb_to_grayscale_paper(img_bgr)
    cropped, bbox, ok = otsu_roi_crop(img_bgr, gray)
    return cropped, bbox, ok, "ROI_OK" if ok else "FALLBACK_FULL"

## 5. Jalankan Pipeline ROI

In [ ]:
output_root    = Path(OUTPUT_ROOT)
log_rows       = []
total_ok = total_fallback = total_error = 0
total_mask_ok = total_mask_err = total_mask_skip = 0

for split_key, src_dir_str in INPUT_SPLITS.items():
    src_dir = Path(src_dir_str)
    dst_dir = output_root / split_key
    dst_dir.mkdir(parents=True, exist_ok=True)

    image_paths = sorted(
        list(src_dir.glob('*.jpg'))  +
        list(src_dir.glob('*.jpeg')) +
        list(src_dir.glob('*.png'))  +
        list(src_dir.glob('*.JPG'))  +
        list(src_dir.glob('*.PNG'))
    )

    if not image_paths:
        print(f"PERINGATAN: tidak ada gambar di {src_dir}")
        continue

    masks_src_dir = src_dir / "masks"
    masks_dst_dir = dst_dir / "masks"
    has_masks     = masks_src_dir.exists() and any(masks_src_dir.glob("*.png"))

    if has_masks:
        masks_dst_dir.mkdir(parents=True, exist_ok=True)

    # Tahap 1: proses semua gambar, simpan bbox untuk pairing mask
    bbox_map: dict[str, tuple | None] = {}

    for src_path in tqdm(image_paths, desc=f"{split_key}", ncols=80):
        dst_path = dst_dir / src_path.name
        output_img, bbox, success, status = process_image(src_path)

        if status == "ERROR_READ":
            total_error += 1
            log_rows.append({'split': split_key, 'file': src_path.name,
                             'status': status, 'note': 'Gagal dibaca oleh OpenCV'})
            continue

        cv2.imwrite(str(dst_path), output_img)

        if success:
            total_ok += 1
        else:
            total_fallback += 1
            log_rows.append({'split': split_key, 'file': src_path.name,
                             'status': status,
                             'note': 'Tidak ada kontur valid / crop terlalu kecil -> full image'})

        if has_masks:
            key = extract_image_pair_key(src_path.stem)
            if key is not None:
                bbox_map[key] = bbox

    # Tahap 2: proses mask dengan bbox dari gambar pasangannya
    if has_masks:
        mask_paths = sorted(masks_src_dir.glob("*.png"))

        mask_keys = {extract_mask_pair_key(mp.stem) for mp in mask_paths} - {None}
        unpaired  = mask_keys - set(bbox_map.keys())
        if unpaired:
            print(f"PERINGATAN [{split_key}]: mask tidak berpasangan: {unpaired}")

        for mask_src_path in mask_paths:
            key = extract_mask_pair_key(mask_src_path.stem)

            if key is None:
                total_mask_err += 1
                log_rows.append({'split': split_key, 'file': mask_src_path.name,
                                 'status': 'MASK_ERROR',
                                 'note': f'Tidak bisa ekstrak key: {mask_src_path.stem}'})
                continue

            if key not in bbox_map:
                total_mask_err += 1
                log_rows.append({'split': split_key, 'file': mask_src_path.name,
                                 'status': 'MASK_NO_IMAGE_PAIR',
                                 'note': f'Gambar pasangan key "{key}" tidak ditemukan'})
                continue

            bbox = bbox_map[key]

            if bbox is None:
                total_mask_skip += 1
                mask_raw = cv2.imread(str(mask_src_path), cv2.IMREAD_GRAYSCALE)
                if mask_raw is not None:
                    cv2.imwrite(str(masks_dst_dir / mask_src_path.name), mask_raw)
                log_rows.append({'split': split_key, 'file': mask_src_path.name,
                                 'status': 'MASK_SKIP',
                                 'note': f'Pasangan [{key}] fallback - mask disalin tanpa crop'})
            else:
                try:
                    cropped_mask = crop_mask_with_bbox(mask_src_path, bbox)
                    cv2.imwrite(str(masks_dst_dir / mask_src_path.name), cropped_mask)
                    total_mask_ok += 1
                except Exception as e:
                    total_mask_err += 1
                    log_rows.append({'split': split_key, 'file': mask_src_path.name,
                                     'status': 'MASK_ERROR', 'note': str(e)})

total = total_ok + total_fallback + total_error
print("SELESAI")
print(f"Total diproses : {total}")
print(f"ROI berhasil   : {total_ok} ({total_ok/total*100:.1f}%)")
print(f"Fallback full  : {total_fallback} ({total_fallback/total*100:.1f}%)")
print(f"Error baca     : {total_error} ({total_error/total*100:.1f}%)")
if total_mask_ok + total_mask_err + total_mask_skip > 0:
    print(f"Mask: {total_mask_ok} OK / {total_mask_err} error / {total_mask_skip} skip")
print(f"Output: {OUTPUT_ROOT}")

if log_rows:
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    log_path  = output_root / f"roi_log_{timestamp}.csv"
    with open(log_path, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=['split', 'file', 'status', 'note'])
        writer.writeheader()
        writer.writerows(log_rows)
    print(f"Log -> {log_path}")

## 6. Verifikasi Visual — Before vs After

In [ ]:
N_SAMPLES    = 5
SAMPLE_SPLIT = 'train/tumor'

src_dir    = Path(INPUT_SPLITS[SAMPLE_SPLIT])
dst_dir    = Path(OUTPUT_ROOT) / SAMPLE_SPLIT
sample_dir = Path(OUTPUT_ROOT) / 'sample_preprocessing'
sample_dir.mkdir(parents=True, exist_ok=True)

samples = (sorted(src_dir.glob('*.jpg')) + sorted(src_dir.glob('*.png')))[:N_SAMPLES]

if not samples:
    print(f"Tidak ada sampel di {src_dir}.")
else:
    fig, axes = plt.subplots(len(samples), 2, figsize=(10, 3 * len(samples)))
    fig.suptitle(f'ROI Preprocessing - Before vs After ({SAMPLE_SPLIT})',
                 fontsize=13, fontweight='bold')

    for i, src_path in enumerate(samples, start=1):
        dst_path = dst_dir / src_path.name
        orig_bgr = cv2.imread(str(src_path))
        roi_bgr  = cv2.imread(str(dst_path)) if dst_path.exists() else orig_bgr

        if orig_bgr is None:
            print(f"PERINGATAN: gagal membaca {src_path.name}")
            continue

        before_path = sample_dir / f"before_{i:02d}_{src_path.stem}.png"
        after_path  = sample_dir / f"after_{i:02d}_{src_path.stem}.png"
        cv2.imwrite(str(before_path), orig_bgr)
        cv2.imwrite(str(after_path),  roi_bgr)

        axes[i-1, 0].imshow(cv2.cvtColor(orig_bgr, cv2.COLOR_BGR2RGB))
        axes[i-1, 0].set_title(f'Before  {orig_bgr.shape[1]}x{orig_bgr.shape[0]}', fontsize=9)
        axes[i-1, 0].axis('off')

        axes[i-1, 1].imshow(cv2.cvtColor(roi_bgr, cv2.COLOR_BGR2RGB))
        axes[i-1, 1].set_title(f'After  {roi_bgr.shape[1]}x{roi_bgr.shape[0]}', fontsize=9)
        axes[i-1, 1].axis('off')

    plt.tight_layout()
    plt.show()
    print(f"Sampel disimpan di: {sample_dir}")

## 7. Verifikasi Visual — Mask + Pasangan Gambar Otak

In [ ]:
MASK_SAMPLES = 5

sample_dir = Path(OUTPUT_ROOT) / 'sample_preprocessing'
sample_dir.mkdir(parents=True, exist_ok=True)

chosen_split = next(
    (s for s in INPUT_SPLITS
     if s.endswith('/tumor')
     and (Path(INPUT_SPLITS[s]) / 'masks').exists()
     and any((Path(INPUT_SPLITS[s]) / 'masks').glob('*.png'))),
    None
)

if chosen_split is None:
    print("PERINGATAN: tidak ditemukan folder 'masks/' pada split tumor manapun.")
else:
    image_src_dir = Path(INPUT_SPLITS[chosen_split])
    mask_src_dir  = image_src_dir / 'masks'
    image_dst_dir = Path(OUTPUT_ROOT) / chosen_split
    mask_dst_dir  = image_dst_dir / 'masks'

    all_images = sorted(image_src_dir.glob('*.jpg')) + sorted(image_src_dir.glob('*.png'))
    mask_paths = sorted(mask_src_dir.glob('*.png'))[:MASK_SAMPLES]

    if not mask_paths:
        print("Tidak ada file mask ditemukan.")
    else:
        fig, axes = plt.subplots(len(mask_paths), 2, figsize=(8, 3 * len(mask_paths)))
        if len(mask_paths) == 1:
            axes = axes.reshape(1, 2)
        fig.suptitle(f'Mask + Gambar Otak ({chosen_split}, hasil ROI)',
                     fontsize=13, fontweight='bold')

        for i, mask_path in enumerate(mask_paths, start=1):
            key = extract_mask_pair_key(mask_path.stem)
            paired_image = next(
                (c for c in all_images if extract_image_pair_key(c.stem) == key),
                None
            ) if key else None

            if paired_image is None:
                print(f"PERINGATAN [{i}]: gambar pasangan tidak ditemukan untuk {mask_path.name}")
                continue

            out_mask_path  = mask_dst_dir  / mask_path.name
            out_image_path = image_dst_dir / paired_image.name

            mask_img  = cv2.imread(str(out_mask_path  if out_mask_path.exists()  else mask_path),
                                   cv2.IMREAD_GRAYSCALE)
            brain_img = cv2.imread(str(out_image_path if out_image_path.exists() else paired_image))

            if mask_img is None or brain_img is None:
                print(f"PERINGATAN [{i}]: gagal membaca mask/gambar key={key}")
                continue

            mask_save_path  = sample_dir / f"mask_{i:02d}_{mask_path.stem}.png"
            brain_save_path = sample_dir / f"brain_{i:02d}_{paired_image.stem}.png"
            cv2.imwrite(str(mask_save_path),  mask_img)
            cv2.imwrite(str(brain_save_path), brain_img)

            axes[i-1, 0].imshow(cv2.cvtColor(brain_img, cv2.COLOR_BGR2RGB))
            axes[i-1, 0].set_title(f'Brain  {brain_img.shape[1]}x{brain_img.shape[0]}', fontsize=9)
            axes[i-1, 0].axis('off')

            axes[i-1, 1].imshow(mask_img, cmap='gray')
            axes[i-1, 1].set_title(f'Mask  {mask_img.shape[1]}x{mask_img.shape[0]}', fontsize=9)
            axes[i-1, 1].axis('off')

        plt.tight_layout()
        plt.show()
        print(f"Sampel disimpan di: {sample_dir}")

## 8. Sanity Check — Shape Gambar & Mask

In [ ]:
img  = cv2.imread('/content/drive/MyDrive/Tugas Akhir/dataset_roi_preprocessed_GC_val_test_FINAL/test/tumor/image_tumor_mask__tumor_mask_8.png')
mask = cv2.imread('/content/drive/MyDrive/Tugas Akhir/dataset_roi_preprocessed_GC_val_test_FINAL/test/tumor/masks/mask_tumor__mask_8.png')

print("Gambar shape:", img.shape)
print("Mask shape  :", mask.shape)